# धडा ०२ - मायक्रोसॉफ्ट एजंट फ्रेमवर्क अन्वेषण

**मायक्रोसॉफ्ट एजंट फ्रेमवर्क (MAF)** हा एआय एजंट तयार करण्यासाठी एक एकत्रित फ्रेमवर्क आहे. तो चार प्रमुख घटकांसह स्वच्छ, रचनात्मक आर्किटेक्चर प्रदान करतो:

- **क्लायंट** – एआय मॉडेल एंडपॉइंटशी कनेक्ट होतो आणि संवाद हाताळतो
- **एजंट** – सूचनांसह आणि साधने व्याख्यांसह क्लायंटला आवळतो
- **साधने** – मॉडेल कॉल करू शकणाऱ्या सानुकूल फंक्शन्ससह एजंट क्षमता वाढवतो
- **सेशन** – मल्टि-टर्न संवादासाठी संभाषण इतिहास ठेवतो

या धड्यात, आपण या संकल्पनांचा वापर करून **प्रवास बुकिंग एजंट** तयार करू, जो गंतव्य स्थळाची उपलब्धता तपासतो.


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## एजंट फ्रेमवर्क आर्किटेक्चर समजून घेणे

Microsoft एजंट फ्रेमवर्क लेयर्ड आर्किटेक्चरचे अनुसरण करतो:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लायंट** – `FoundryChatClient` Azure OpenAI डिप्लॉयमेंटशी कनेक्ट होतो. तो प्रमाणीकरण, विनंती फॉर्मॅटिंग आणि प्रतिसाद पार्सिंग हाताळतो.
2. **एजंट** – क्लायंट कडून `provider.create_agent()` द्वारे तयार केला जातो, एजंट मॉडेल प्रवेश, सूचना (सिस्टम प्रॉम्प्ट) आणि टूल्स यांना एकत्र करतो.
3. **टूल्स** – `@tool` ने डेकोरेट केलेली Python फंक्शन्स जी एजंट कार्ये पूर्ण करण्यासाठी किंवा डेटा मिळवण्यासाठी कॉल करू शकतो.
4. **सत्र** – `AgentSession` ऑब्जेक्ट (जो `agent.create_session()` द्वारे तयार होतो) जो संवाद इतिहास साठवतो, बहु-टर्न संवाद सक्षम करतो जिथे एजंट पूर्वीचा संदर्भ लक्षात ठेवतो.

चला प्रत्येक लेयर टप्प्याटप्प्याने तयार करूया.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool डेकोरेटरसह टूल्स जोडणे

टूल्स एजंट्सना फक्त मजकूर तयार करण्यापलीकडे कृती करण्याची परवानगी देतात. `@tool` डेकोरेटर सामान्य पायथन फंक्शनला अशा काहीतरी स्वरूपात रूपांतरित करतो जे एजंट कॉल करू शकतो.

मुख्य मुद्दे:
- मॉडेल प्रत्येक पॅरामीटर समजून घेईल यासाठी `Annotated[type, "description"]` वापरा.
- डॉकस्ट्रिंग टूलचे वर्णन बनते जे मॉडेल पाहते.
- `approval_mode="never_require"` म्हणजे टूल वापरकर्त्याच्या पुष्टीशिवाय आपोआप चालते.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## साधने वापरून एजंट तयार करणे

आता आपण क्लायंट, सूचना आणि साधने एकत्र करून एजंट तयार करू. `सूचना` प्रणाली संकेत म्हणून कार्य करतात — त्या एजंटच्या व्यक्तिमत्व आणि वर्तनाची व्याख्या करतात.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रांसह मल्टी-टर्न संभाषणे

एक `AgentSession` (जो `agent.create_session()` द्वारे तयार केला जातो) संभाषणातील सर्व संदेशांचे ट्रॅक ठेवतो. प्रत्येक `agent.run()` कॉलमध्ये तोच सत्र देऊन, एजंटला पूर्ण संभाषणाचा इतिहास मिळतो आणि तो मागील संदेशांकडे परत पाहू शकतो.

आम्ही `tools=[check_destination_availability]` पास करतो जेणेकरून एजंट प्रत्येक टर्नमध्ये आमचा उपलब्धता तपासणी करणारा कॉल करू शकेल.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

या धड्यामध्ये आपण मायक्रोसॉफ्ट एजंट फ्रेमवर्कच्या चार पाया संशोधित केले:

| संकल्पना | आपण काय शिकलात |
|---------|------------------|
| **क्लायंट** | `FoundryChatClient` क्रेडेन्शियल-आधारित प्रमाणीकरणासह Azure OpenAI शी कनेक्ट होतो |
| **एजंट** | `provider.create_agent()` मॉडेल कनेक्शनसह सूचना आणि नाव एकत्रित करतो |
| **टूल्स** | `@tool` डेकोरेटर एजंटला कॉल करण्यासाठी Python फंक्शन्स सार्वजनिक करते |
| **सेशन** | `agent.create_session()` अनेक संवाद टप्प्यांमध्ये संभाषण इतिहास राखतो |

हे बांधकाम घटक एकत्र करून असे एजंट तयार करतात जे नैसर्गिक संवाद ठेवू शकतात, बाह्य फंक्शन्स कॉल करू शकतात, आणि संदर्भ राखू शकतात — पुढील धड्यांमध्ये अधिक प्रगत एजंटिक पॅटर्नसाठी पाया.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
